In [102]:
import pandas as pd

population_df = pd.read_excel(r'data\1.2-folkmangd-den-31-dec-2010-2034.xlsx')

population_df.head()

,Folkmängd den 31 dec 2010-2034,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2010.0,2015.0,2020.0,2024.0,NaN,2034¹,NaN,"Fördelning, %",NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,2034¹
3,Västerort,213629.0,235101.0,247170.0,254678.0,NaN,271826.538978,NaN,25.581022,25.700278
4,Järva,83027.0,86472.0,89847.0,95220.0,NaN,104610.170555,NaN,9.564332,9.890537


In [103]:
population_df = population_df.dropna(how='all')
population_df = population_df.dropna(axis=1, how='all')
population_df = population_df.iloc[3:17]
population_df = population_df.iloc[:, [0, 4]]
population_df.columns = ['area', 'population_2024']
population_df['population_2024'] = population_df['population_2024'].round(0).astype(int)
population_df.head()

,area,population_2024
3,Västerort,254678
4,Järva,95220
5,Hässelby-Vällingby,76340
6,Bromma,83118
8,Inre staden,356620


In [104]:
import pandas as pd

crime_df = pd.read_excel(r'data\1.63-anmalda-brott-2024.xlsx')

crime_df.head()

,Anmälda brott 2024. Stockholm,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,Number of reported crimes in 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,An-,NaN,Därav,NaN,NaN,NaN,NaN,Brott per 1 000 invånare,NaN,NaN,NaN,NaN
2,NaN,mälda,NaN,Vålds-,Stöld,Skade-,Narkotika-,NaN,An-,Vålds-,Stöld,Skade-,Narkotika-
3,NaN,brott1,NaN,brott2,& rån3,görelse,brott4,NaN,mälda,brott,& rån,görelse,brott
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brott,NaN,NaN,NaN,NaN


In [105]:
crime_df = crime_df.dropna(how='all')
crime_df = crime_df.dropna(axis=1, how='all')
crime_df = crime_df.iloc[5:19]
crime_df = crime_df.iloc[:, [0, 6]]
crime_df.columns = ['area', 'reported_crime_per_1000_2024']
crime_df['reported_crime_per_1000_2024'] = pd.to_numeric(crime_df['reported_crime_per_1000_2024'])
crime_df['reported_crime_per_1000_2024'] = crime_df['reported_crime_per_1000_2024'].round(0).astype(int)
crime_df.head()

,area,reported_crime_per_1000_2024
5,Västerort,180
6,Järva,168
7,Hässelby-Vällingby,209
8,Bromma,167
10,Inre staden,238


In [106]:
district_df = pd.DataFrame({
    'areaID': range(1, crime_df['area'].nunique() + 1),
    'area': crime_df['area'].unique(),
})

district_df.head()

,areaID,area
0,1,Västerort
1,2,Järva
2,3,Hässelby-Vällingby
3,4,Bromma
4,5,Inre staden


In [107]:
merged_crime_df = crime_df.merge(district_df, left_on='area', right_on='area')

merged_crime_df = merged_crime_df.drop(columns='area')

merged_crime_df.head()

,reported_crime_per_1000_2024,areaID
0,180,1
1,168,2
2,209,3
3,167,4
4,238,5


In [108]:
merged_population_df = population_df.merge(district_df, left_on='area', right_on='area')

merged_population_df = merged_population_df.drop(columns='area')

merged_population_df.head()

,population_2024,areaID
0,254678,1
1,95220,2
2,76340,3
3,83118,4
4,356620,5


In [ ]:
import pandas as pd
import duckdb

con = duckdb.connect()
con.register('population', merged_population_df)
con.register('crime', merged_crime_df)
con.register('districts', district_df)

result = con.execute("""--sql
                    SELECT
                     d.area,
                     p.population_2024
                    FROM population p
                    INNER JOIN districts d ON p.areaID = d.areaID
                    WHERE d.area = 'Järva'""").fetch_df()

result

,area,population_2024
0,Järva,95220
